# 04 — Customer forecasting

Run from a fresh kernel inside the P6 project. Requires pandas, numpy, scikit-learn and joblib. The final merge requires `predictions/test_predictions_detailed.csv` from notebook 02. This notebook fits inexpensive median baselines; it does not train CatBoost or LSTM.

Existing displayed results are historical; edited cells have cleared outputs.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from IPython.display import display
current_dir = Path.cwd().resolve()
project_dir = next((p for p in [current_dir, *current_dir.parents]
                    if (p / "data" / "train.csv").is_file()), None)
if project_dir is None:
    raise FileNotFoundError("Open this notebook inside the P6 project containing data/train.csv")
train_path = project_dir / "data" / "train.csv"

customer_data = pd.read_csv(
    train_path,
    usecols=[
        "Store", "Date", "DayOfWeek", "Customers",
        "Open", "Promo", "StateHoliday", "SchoolHoliday"
    ],
    parse_dates=["Date"],
    dtype={"StateHoliday": "string"},
).sort_values(["Store", "Date"]).reset_index(drop=True)

assert not customer_data.duplicated(["Store", "Date"]).any()
assert customer_data["Customers"].notna().all()
assert (customer_data["Customers"] >= 0).all()

validation_start = pd.Timestamp("2015-06-20")

customer_train = customer_data.loc[
    customer_data["Date"] < validation_start
].copy()

customer_valid = customer_data.loc[
    customer_data["Date"] >= validation_start
].copy()

print("Customer dataset:", customer_data.shape)
print("Stores:", customer_data["Store"].nunique())
print(
    "Training period:",
    customer_train["Date"].min().date(),
    "to", customer_train["Date"].max().date()
)
print(
    "Validation period:",
    customer_valid["Date"].min().date(),
    "to", customer_valid["Date"].max().date()
)
print("Validation rows:", len(customer_valid))
print("Missing values:")
print(customer_data.isna().sum())
print("\nCustomer forecasting data prepared.")

Customer dataset: (1017209, 8)
Stores: 1115
Training period: 2013-01-01 to 2015-06-19
Validation period: 2015-06-20 to 2015-07-31
Validation rows: 46830
Missing values:
Store            0
DayOfWeek        0
Date             0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

Customer forecasting data prepared.


In [2]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Fit customer baselines using training data only
customer_train_open = customer_train.loc[
    customer_train["Open"].eq(1)
].copy()

customer_keys = ["Store", "DayOfWeek", "Promo"]

customer_lookup = (
    customer_train_open.groupby(customer_keys)["Customers"]
    .median()
    .rename("Grouped_prediction")
    .reset_index()
)

customer_store_medians = (
    customer_train_open.groupby("Store")["Customers"].median()
)
customer_global_median = float(
    customer_train_open["Customers"].median()
)

customer_validation_results = customer_valid.merge(
    customer_lookup,
    on=customer_keys,
    how="left",
    validate="many_to_one",
)

customer_validation_results["Store_prediction"] = (
    customer_validation_results["Store"]
    .map(customer_store_medians)
    .fillna(customer_global_median)
)

customer_validation_results["Grouped_prediction"] = (
    customer_validation_results["Grouped_prediction"]
    .fillna(customer_validation_results["Store_prediction"])
)

prediction_columns = ["Store_prediction", "Grouped_prediction"]

# Known closed days receive zero predictions
customer_validation_results.loc[
    customer_validation_results["Open"].eq(0),
    prediction_columns,
] = 0.0

assert len(customer_validation_results) == len(customer_valid)
assert np.isfinite(
    customer_validation_results[prediction_columns].to_numpy()
).all()

def customer_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    error = predicted - actual
    positive = actual > 0

    return {
        "MAE": np.abs(error).mean(),
        "RMSE": np.sqrt(np.mean(error ** 2)),
        "RMSPE_pct": (
            100 * np.sqrt(np.mean(
                (error[positive] / actual[positive]) ** 2
            ))
            if positive.any() else np.nan
        ),
    }

customer_baseline_metrics = pd.DataFrame({
    "Store median": customer_metrics(
        customer_validation_results["Customers"],
        customer_validation_results["Store_prediction"],
    ),
    "Store–weekday–promotion median": customer_metrics(
        customer_validation_results["Customers"],
        customer_validation_results["Grouped_prediction"],
    ),
}).T

display(customer_baseline_metrics.round(2))

tables_dir = project_dir / "reports" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)

customer_baseline_metrics.to_csv(
    tables_dir / "customer_baseline_metrics.csv",
    index_label="Model",
)
customer_validation_results.to_csv(
    tables_dir / "customer_baseline_validation_predictions.csv",
    index=False,
)

print("Customer baseline metrics and predictions saved.")
print("MAE/RMSE: all validation rows; RMSPE: positive customer counts only.")

,MAE,RMSE,RMSPE_pct
Store median,94.19,143.97,28.89
Store–weekday–promotion median,56.93,93.74,12.71


Customer baseline metrics and predictions saved.
MAE/RMSE: all validation rows; RMSPE: positive customer counts only.


In [3]:
# Earlier customer backtest: 9 May–19 June 2015
earlier_start = pd.Timestamp("2015-05-09")
earlier_end = pd.Timestamp("2015-06-19")

earlier_customer_train = customer_data.loc[
    (customer_data["Date"] < earlier_start)
    & customer_data["Open"].eq(1)
].copy()

earlier_customer_valid = customer_data.loc[
    customer_data["Date"].between(earlier_start, earlier_end)
].copy()

earlier_lookup = (
    earlier_customer_train
    .groupby(customer_keys)["Customers"]
    .median()
    .rename("Grouped_prediction")
    .reset_index()
)

earlier_store_medians = (
    earlier_customer_train.groupby("Store")["Customers"].median()
)
earlier_global_median = float(
    earlier_customer_train["Customers"].median()
)

earlier_customer_results = earlier_customer_valid.merge(
    earlier_lookup,
    on=customer_keys,
    how="left",
    validate="many_to_one",
)

earlier_customer_results["Store_prediction"] = (
    earlier_customer_results["Store"]
    .map(earlier_store_medians)
    .fillna(earlier_global_median)
)

earlier_customer_results["Grouped_prediction"] = (
    earlier_customer_results["Grouped_prediction"]
    .fillna(earlier_customer_results["Store_prediction"])
)

earlier_customer_results.loc[
    earlier_customer_results["Open"].eq(0),
    ["Store_prediction", "Grouped_prediction"],
] = 0.0

assert len(earlier_customer_results) == len(earlier_customer_valid)
assert np.isfinite(
    earlier_customer_results[
        ["Store_prediction", "Grouped_prediction"]
    ].to_numpy()
).all()

earlier_customer_metrics = pd.DataFrame({
    "Store median": customer_metrics(
        earlier_customer_results["Customers"],
        earlier_customer_results["Store_prediction"],
    ),
    "Store–weekday–promotion median": customer_metrics(
        earlier_customer_results["Customers"],
        earlier_customer_results["Grouped_prediction"],
    ),
}).T

print("Earlier customer backtest: 9 May–19 June 2015")
print("Evaluation rows:", len(earlier_customer_results))
display(earlier_customer_metrics.round(2))

earlier_customer_metrics.to_csv(
    tables_dir / "customer_earlier_backtest_metrics.csv",
    index_label="Model",
)
earlier_customer_results.to_csv(
    tables_dir / "customer_earlier_backtest_predictions.csv",
    index=False,
)

print("Earlier customer backtest saved.")

Earlier customer backtest: 9 May–19 June 2015
Evaluation rows: 46830


,MAE,RMSE,RMSPE_pct
Store median,85.11,136.57,26.71
Store–weekday–promotion median,52.41,89.87,12.69


Earlier customer backtest saved.


In [4]:
import joblib
import json
from datetime import datetime, timezone

customer_full_open = customer_data.loc[
    customer_data["Open"].eq(1)
].copy()

customer_bundle = {
    "model_type": "Store-weekday-promotion median",
    "keys": ["Store", "DayOfWeek", "Promo"],
    "lookup": (
        customer_full_open
        .groupby(["Store", "DayOfWeek", "Promo"])["Customers"]
        .median()
    ),
    "store_medians": (
        customer_full_open.groupby("Store")["Customers"].median()
    ),
    "global_median": float(customer_full_open["Customers"].median()),
    "training_end": str(customer_data["Date"].max().date()),
}

def predict_customers(frame, bundle):
    keys = bundle["keys"]
    required = keys + ["Open"]

    if frame[required].isna().any().any():
        raise ValueError("Customer prediction inputs contain missing values.")
    if not frame["Open"].isin([0, 1]).all():
        raise ValueError("Open must contain only 0 or 1.")

    lookup_index = pd.MultiIndex.from_frame(frame[keys])
    predictions = pd.Series(
        bundle["lookup"].reindex(lookup_index).to_numpy(),
        index=frame.index,
        dtype=float,
    )

    predictions = (
        predictions
        .fillna(frame["Store"].map(bundle["store_medians"]))
        .fillna(bundle["global_median"])
        .clip(lower=0)
    )
    predictions.loc[frame["Open"].eq(0)] = 0.0
    return predictions.to_numpy()

models_dir = project_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

customer_model_path = models_dir / "customer_baseline_final.joblib"
joblib.dump(customer_bundle, customer_model_path)

# Confirm that saving and reloading preserves predictions
loaded_customer_bundle = joblib.load(customer_model_path)
check_rows = customer_data.sample(n=1000, random_state=42)

np.testing.assert_allclose(
    predict_customers(check_rows, customer_bundle),
    predict_customers(check_rows, loaded_customer_bundle),
)

customer_metadata = {
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_file": customer_model_path.name,
    "model_type": customer_bundle["model_type"],
    "training_end": customer_bundle["training_end"],
    "training_open_rows": len(customer_full_open),
    "prediction_inputs": ["Store", "DayOfWeek", "Promo", "Open"],
    "fallback_order": [
        "Store-weekday-promotion median",
        "Store median",
        "Global median",
    ],
    "closed_day_rule": "Predict zero when Open=0",
    "earlier_backtest": json.loads(
        earlier_customer_metrics.to_json(orient="index")
    ),
    "later_backtest": json.loads(
        customer_baseline_metrics.to_json(orient="index")
    ),
    "limitations": [
        "Does not explicitly model holiday effects or changing trends.",
        "Backtests cover 42 days; final test horizon covers 48 days.",
        "Backtests informed model selection; they are development results.",
        "Predictions are estimates and may contain fractional customer counts.",
    ],
}

(models_dir / "customer_model_metadata.json").write_text(
    json.dumps(customer_metadata, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Final customer baseline saved:", customer_model_path)
print("Reload check passed: predictions match.")
print("Customer model metadata saved.")

Final customer baseline saved: D:\DigiCrome Academy\Internship\Nexthikes\P6\models\customer_baseline_final.joblib
Reload check passed: predictions match.
Customer model metadata saved.


In [5]:
test_path = project_dir / "data" / "test.csv"

customer_test = pd.read_csv(
    test_path,
    usecols=["Id", "Store", "Date", "DayOfWeek", "Promo", "Open"],
    parse_dates=["Date"],
)

assert customer_test["Id"].is_unique
assert not customer_test.duplicated(["Store", "Date"]).any()

# Use the same missing-Open assumption as the sales forecast
customer_test["OpenWasMissing"] = customer_test["Open"].isna()
customer_test["Open"] = customer_test["Open"].fillna(1)

customer_test["Customers_prediction"] = predict_customers(
    customer_test,
    loaded_customer_bundle,
)

assert np.isfinite(customer_test["Customers_prediction"]).all()
assert customer_test["Customers_prediction"].ge(0).all()
assert customer_test.loc[
    customer_test["Open"].eq(0), "Customers_prediction"
].eq(0).all()

predictions_dir = project_dir / "predictions"
predictions_dir.mkdir(parents=True, exist_ok=True)

customer_forecast_path = (
    predictions_dir / "customer_test_predictions.csv"
)
customer_test.to_csv(customer_forecast_path, index=False)

print("Customer forecast rows:", len(customer_test))
print(
    "Forecast period:",
    customer_test["Date"].min().date(),
    "to", customer_test["Date"].max().date(),
)
print("Missing Open values filled with 1:",
      customer_test["OpenWasMissing"].sum())
print("Customer forecasts saved:", customer_forecast_path)

display(
    customer_test[
        ["Id", "Store", "Date", "Open", "Customers_prediction"]
    ].head(10)
)

Customer forecast rows: 41088
Forecast period: 2015-08-01 to 2015-09-17
Missing Open values filled with 1: 11
Customer forecasts saved: D:\DigiCrome Academy\Internship\Nexthikes\P6\predictions\customer_test_predictions.csv


,Id,Store,Date,Open,Customers_prediction
0,1,1,2015-09-17,1.0,543.0
1,2,3,2015-09-17,1.0,815.5
2,3,7,2015-09-17,1.0,960.0
3,4,8,2015-09-17,1.0,834.0
4,5,9,2015-09-17,1.0,586.0
5,6,10,2015-09-17,1.0,606.0
6,7,11,2015-09-17,1.0,1137.0
7,8,12,2015-09-17,1.0,920.0
8,9,13,2015-09-17,1.0,405.5
9,10,14,2015-09-17,1.0,667.0


In [6]:
sales_path = predictions_dir / "test_predictions_detailed.csv"
if not sales_path.is_file():
    raise FileNotFoundError("Run notebook 02 or restore predictions/test_predictions_detailed.csv before this final merge. Customer forecasts have already been saved.")

sales_forecasts = pd.read_csv(
    predictions_dir / "test_predictions_detailed.csv",
    parse_dates=["Date"],
)

customer_forecasts = pd.read_csv(
    predictions_dir / "customer_test_predictions.csv",
    parse_dates=["Date"],
)

join_keys = ["Id", "Store", "Date"]

assert sales_forecasts["Id"].is_unique
assert customer_forecasts["Id"].is_unique

dashboard_forecasts = sales_forecasts.merge(
    customer_forecasts[
        join_keys + ["Customers_prediction"]
    ],
    on=join_keys,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

assert dashboard_forecasts["_merge"].eq("both").all(), (
    "Sales and customer forecast records do not match."
)
dashboard_forecasts = dashboard_forecasts.drop(columns="_merge")

assert len(dashboard_forecasts) == 41088
assert np.isfinite(
    dashboard_forecasts[["Sales", "Customers_prediction"]].to_numpy()
).all()
assert (
    dashboard_forecasts[["Sales", "Customers_prediction"]] >= 0
).all().all()

closed_rows = dashboard_forecasts["Open"].eq(0)
assert dashboard_forecasts.loc[
    closed_rows, ["Sales", "Customers_prediction"]
].eq(0).all().all()

dashboard_forecasts = dashboard_forecasts.sort_values(
    "Id"
).reset_index(drop=True)

dashboard_path = predictions_dir / "dashboard_forecasts.csv"
dashboard_forecasts.to_csv(dashboard_path, index=False)

print("Combined forecast rows:", len(dashboard_forecasts))
print("Sales and customer records matched successfully.")
print("Dashboard forecasts saved:", dashboard_path)

display(
    dashboard_forecasts[
        ["Id", "Store", "Date", "Sales", "Customers_prediction"]
    ].head(10)
)

Combined forecast rows: 41088
Sales and customer records matched successfully.
Dashboard forecasts saved: D:\DigiCrome Academy\Internship\Nexthikes\P6\predictions\dashboard_forecasts.csv


,Id,Store,Date,Sales,Customers_prediction
0,1,1,2015-09-17,4587.243591,543.0
1,2,3,2015-09-17,7805.857665,815.5
2,3,7,2015-09-17,9687.286745,960.0
3,4,8,2015-09-17,6554.353397,834.0
4,5,9,2015-09-17,6749.779523,586.0
5,6,10,2015-09-17,5719.217987,606.0
6,7,11,2015-09-17,8311.944033,1137.0
7,8,12,2015-09-17,7976.887273,920.0
8,9,13,2015-09-17,5431.020725,405.5
9,10,14,2015-09-17,5697.509995,667.0


In [7]:
from pathlib import Path

project = Path(r"D:\DigiCrome Academy\Internship\Nexthikes\P6")

for name in [
    "submission.csv",
    "test_predictions_detailed.csv",
    "customer_test_predictions.csv",
    "dashboard_forecasts.csv",
]:
    path = project / "predictions" / name
    print(f"{'FOUND' if path.is_file() else 'MISSING'}: {name}")

FOUND: submission.csv
FOUND: test_predictions_detailed.csv
FOUND: customer_test_predictions.csv
FOUND: dashboard_forecasts.csv


In [10]:
from pathlib import Path
import subprocess

file = Path(
    r"D:\DigiCrome Academy\Internship\Nexthikes\P6"
) / "predictions" / "dashboard_forecasts.csv"

print("File exists:", file.is_file())
print("Location:", file)

if file.is_file():
    subprocess.run(["explorer.exe", "/select,", str(file)])

File exists: True
Location: D:\DigiCrome Academy\Internship\Nexthikes\P6\predictions\dashboard_forecasts.csv


In [11]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

st.set_page_config(
    page_title="Rossmann Forecast Dashboard",
    page_icon="📊",
    layout="wide",
)

PROJECT = Path(__file__).resolve().parent
FORECAST_FILE = PROJECT / "predictions" / "dashboard_forecasts.csv"


@st.cache_data
def load_forecasts(path, modified_time):
    return pd.read_csv(path, parse_dates=["Date"])


st.title("Rossmann Sales & Customer Forecasts")
st.caption(
    "Saved predictions for 1 August–17 September 2015. "
    "These are forecast values, not actual results."
)

if not FORECAST_FILE.is_file():
    st.error(f"File not found: {FORECAST_FILE}")
    st.stop()

df = load_forecasts(
    str(FORECAST_FILE),
    FORECAST_FILE.stat().st_mtime_ns,
)

st.sidebar.header("Filter forecasts")

stores = sorted(df["Store"].unique().tolist())
selected_store = st.sidebar.selectbox(
    "Store",
    ["All stores"] + stores,
)

date_range = st.sidebar.date_input(
    "Date range",
    value=(df["Date"].min().date(), df["Date"].max().date()),
    min_value=df["Date"].min().date(),
    max_value=df["Date"].max().date(),
)

if len(date_range) != 2:
    st.info("Select both a start date and an end date.")
    st.stop()

start, end = map(pd.Timestamp, date_range)
filtered = df.loc[df["Date"].between(start, end)].copy()

if selected_store != "All stores":
    filtered = filtered.loc[
        filtered["Store"].eq(selected_store)
    ].copy()

if filtered.empty:
    st.info("No forecasts match these filters.")
    st.stop()

col1, col2, col3 = st.columns(3)

col1.metric(
    "Forecast sales",
    f"{filtered['Sales'].sum():,.0f}",
)
col2.metric(
    "Forecast customer visits",
    f"{filtered['Customers_prediction'].sum():,.0f}",
)
col3.metric(
    "Stores selected",
    f"{filtered['Store'].nunique():,}",
)

st.caption(
    "Customer totals represent estimated visits across store-days, "
    "not unique people."
)

daily = (
    filtered.groupby("Date", as_index=False)[
        ["Sales", "Customers_prediction"]
    ]
    .sum()
    .sort_values("Date")
)

sales_tab, customers_tab, records_tab = st.tabs(
    ["Sales forecast", "Customer forecast", "Forecast records"]
)

with sales_tab:
    fig = px.line(
        daily,
        x="Date",
        y="Sales",
        title="Daily forecast sales",
        labels={"Sales": "Forecast sales"},
    )
    st.plotly_chart(fig, use_container_width=True)

with customers_tab:
    fig = px.line(
        daily,
        x="Date",
        y="Customers_prediction",
        title="Daily forecast customer visits",
        labels={"Customers_prediction": "Forecast customer visits"},
    )
    st.plotly_chart(fig, use_container_width=True)

with records_tab:
    columns = [
        "Id", "Store", "Date", "Open",
        "Promo", "Sales", "Customers_prediction",
    ]
    records = filtered[columns].sort_values(["Date", "Store"])
    st.dataframe(records, use_container_width=True, hide_index=True)

    st.download_button(
        "Download filtered forecasts",
        data=records.to_csv(index=False).encode("utf-8"),
        file_name="filtered_forecasts.csv",
        mime="text/csv",
    )

2026-11-17 13:53:26.715 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name '__file__' is not defined

In [5]:
%%writefile "D:/DigiCrome Academy/Internship/Nexthikes/P6/app.py"
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st

Writing D:/DigiCrome Academy/Internship/Nexthikes/P6/app.py
